# Bài 18: Lưu trữ bài viết với local storage

## Tại sao cần lưu trữ?

Ở các bài trước, khi agent tạo xong một bài viết, kết quả chỉ hiện lên màn hình rồi **biến mất**. Tắt chương trình là mất sạch.

Trong thực tế, chúng ta cần:
- **Lưu trữ** các bài viết đã tạo
- **Theo dõi** metadata (chủ đề, từ khóa, trạng thái, số từ)
- **Tìm kiếm** và **lấy lại** bài viết theo ID
- **Cập nhật** bài viết sau khi tạo (ví dụ: thêm hình ảnh)

Sản phẩm của chúng ta sử dụng **local file storage** (lưu trữ file cục bộ) — đơn giản, không cần dịch vụ bên ngoài.

## Local file — database đơn giản nhất

Cách lưu trữ của chúng ta:
- `content/articles.json` — metadata của tất cả bài viết (chủ đề, từ khóa, trạng thái, số từ)
- `content/{id}.md` — nội dung Markdown đầy đủ của mỗi bài viết

So sánh:
- **Không lưu trữ**: Dữ liệu mất khi tắt chương trình
- **Có local storage**: Dữ liệu lưu vào đĩa, tồn tại qua các lần khởi động lại

Tại sao chọn local file?
- **Không cần thiết lập** — không API key, không tài khoản cloud, không database server
- **Đơn giản** — JSON và Markdown là định dạng mà người có thể đọc được
- **Nhanh** — không gọi mạng, chỉ đọc/ghi đĩa
- **Dễ sao lưu** — copy thư mục `content/` là backup được toàn bộ

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output/backend"))

from dotenv import load_dotenv
load_dotenv()

## Article ID — keyword slug

Mỗi bài viết có một ID duy nhất dựa trên từ khóa hoặc chủ đề.

Thay vì ID trừu tượng như `recABC123` hay số tự tăng như `1, 2, 3`, ID của chúng ta là **keyword slug** (chuỗi từ khóa viết liền bằng gạch ngang):

| Chủ đề | Article ID (slug) |
|-------|------------------|
| "On-Page SEO Meta Tags" | `on-page-seo-meta-tags` |
| "How to Write Better Content" | `how-to-write-better-content` |
| "Technical SEO Guide 2026" | `technical-seo-guide-2026` |

Slug được tạo từ keywords (hoặc topic nếu không có keywords). Điều này làm article ID **dễ đọc** — nhìn ID là biết bài viết về gì.

Nếu slug đã tồn tại, số được nối thêm: `on-page-seo-2`, `on-page-seo-3`, v.v.

## Tầng lưu trữ (storage layer)

File `tools/storage.py` cung cấp 4 hàm dành cho agent:

| Hàm | Mục đích |
|----------|--------|
| `save_article(topic, article_markdown, keywords)` | Lưu bài viết mới vào đĩa |
| `list_all_articles(status_filter)` | Liệt kê tất cả bài viết (chỉ metadata) |
| `get_article_content(article_id)` | Lấy một bài viết kèm nội dung đầy đủ |
| `update_article_content(article_id, article_markdown)` | Cập nhật nội dung bài viết |

Tất cả các hàm đều trả về **chuỗi JSON** — đây là cách Agno tools hoạt động. Agent đọc JSON, hàm trả về JSON.

Hãy import và sử dụng chúng:

In [ ]:
from tools.storage import (
    save_article,
    list_all_articles,
    get_article_content,
    update_article_content,
)

print("Đã import các hàm storage!")
print("  save_article(topic, article_markdown, keywords) -> JSON")
print("  list_all_articles(status_filter) -> JSON")
print("  get_article_content(article_id) -> JSON")
print("  update_article_content(article_id, article_markdown) -> JSON")
print()
print(f"Th\u01b0 m\u1ee5c l\u01b0u tr\u1eef: content/")
print(f"File metadata: content/articles.json")
print(f"File b\u00e0i vi\u1ebft: content/{{id}}.md")

## Lưu bài viết

`save_article()` nhận vào chủ đề, nội dung Markdown, và từ khóa (tùy chọn). Nó sẽ:
1. Tạo slug ID từ keywords
2. Ghi Markdown vào `content/{id}.md`
3. Thêm metadata vào `content/articles.json`
4. Trả về JSON với article ID và thông tin chi tiết

In [ ]:
import json

# Lưu một bài viết thử nghiệm
result = save_article(
    topic="Test Article from Notebook",
    article_markdown="# Test Article\n\nThis is a test article created from the notebook.\n\n## Section 1\n\nSome content here.",
    keywords="test, notebook",
)

article_info = json.loads(result)
print("Đã l\u01b0u b\u00e0i vi\u1ebft:")
print(f"  ID: {article_info['article_id']}")
print(f"  Ch\u1ee7 \u0111\u1ec1: {article_info['topic']}")
print(f"  File: {article_info['filename']}")
print(f"  S\u1ed1 t\u1eeb: {article_info['word_count']}")
print(f"  Tr\u1ea1ng th\u00e1i: {article_info['status']}")
print()
print(f"L\u01b0u \u00fd: ID l\u00e0 slug ('test-notebook'), kh\u00f4ng ph\u1ea3i chu\u1ed7i ng\u1eabu nhi\u00ean hay s\u1ed1 nguy\u00ean.")

## Liệt kê tất cả bài viết

`list_all_articles()` trả về metadata của tất cả bài viết (không tải nội dung đầy đủ — sẽ chậm nếu có nhiều bài).

In [ ]:
result = list_all_articles()
articles = json.loads(result)

print(f"T\u1ed5ng s\u1ed1 b\u00e0i vi\u1ebft: {len(articles)}\n")
for a in articles:
    print(f"  {a['id']}: {a['topic']}")
    print(f"    Tr\u1ea1ng th\u00e1i: {a['status']}, S\u1ed1 t\u1eeb: {a.get('word_count', 'N/A')}")

## Lấy nội dung bài viết

`get_article_content()` tải toàn bộ nội dung Markdown từ file `.md`.

In [ ]:
# Lấy nội dung bài viết thử nghiệm
article_id = article_info['article_id']  # Từ lệnh save ở trên

result = get_article_content(article_id)
content = json.loads(result)

print(f"B\u00e0i vi\u1ebft: {content['topic']}")
print(f"\nN\u1ed9i dung:\n{content['article_markdown']}")

## Cập nhật nội dung bài viết

`update_article_content()` thay thế nội dung Markdown và tính lại số từ. Đây là cách Image Finder thêm hình ảnh — nó đọc bài viết, chèn link hình, rồi lưu lại phiên bản cập nhật.

In [ ]:
# Cập nhật bài viết với nội dung mới
updated_markdown = """# Test Article

This is the updated version with more content.

## Section 1: Introduction

This article was created and then updated from a Jupyter notebook.

## Section 2: Key Points

- Point one: storage is simple
- Point two: local files work great
- Point three: no external services needed

## Conclusion

Local file storage is perfect for this kind of project.
"""

result = update_article_content(article_id, updated_markdown)
updated = json.loads(result)
print(f"Đã cập nhật bài viết {updated['article_id']}")
print(f"Số từ mới: {updated['word_count']}")

## Xem file gốc

Vì bài viết được lưu dưới dạng file thuần, bạn có thể xem trực tiếp:

In [ ]:
# Đọc file metadata articles.json
metadata_path = os.path.abspath("../../content/articles.json")

if os.path.exists(metadata_path):
    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    
    print("content/articles.json:")
    print(json.dumps(metadata, indent=2, ensure_ascii=False)[:2000])
else:
    print("Ch\u01b0a c\u00f3 file articles.json.")

## Thread safety

Module storage sử dụng `threading.Lock` để ngăn chặn hỏng dữ liệu khi nhiều agent lưu bài viết cùng lúc (trong quá trình xử lý batch).

```python
_lock = threading.Lock()

with _lock:
    # Chỉ một thread được vào đây tại một thời điểm
    metadata = _load_metadata()
    metadata[article_id] = {...}
    _save_metadata(metadata)
```

Nếu không có lock, hai agent lưu cùng lúc có thể ghi đè dữ liệu của nhau. Lock đảm bảo một agent hoàn thành trước khi agent khác bắt đầu.

Bạn không cần lo về điều này khi sử dụng các hàm — lock được xử lý nội bộ.

## SQLite cho chat memory

Local file storage xử lý **bài viết**. Nhưng sản phẩm còn sử dụng **SQLite** cho một việc: **lưu lịch sử chat**.

Trong `chat.py` và `team.py`:
```python
db=SqliteDb(db_file="chat_sessions.db")
```

Đây là cơ chế lưu trữ hội thoại có sẵn của Agno. Nó tách biệt với việc lưu bài viết — SQLite lo chat memory, local file lo bài viết.

## Tổng kết

- **Local file storage** lưu bài viết dưới dạng file `.md` với metadata JSON
- **Article ID là keyword slug** — dễ đọc, như `on-page-seo-meta-tags`
- **4 hàm chính**: `save_article()`, `list_all_articles()`, `get_article_content()`, `update_article_content()`
- **Tất cả trả về chuỗi JSON** — định dạng chuẩn cho agent tools
- **Thread-safe** — `threading.Lock` ngăn hỏng dữ liệu khi xử lý batch
- **SQLite** chỉ dùng cho chat memory, không phải cho bài viết

**Bài tiếp theo:** Cách mọi thứ kết nối — theo dõi một yêu cầu từ giao diện chat qua team đến storage.

## Bài tập

Sử dụng các hàm storage đã import ở trên, viết code để:

1. Tạo một bài viết mới với chủ đề và từ khóa tự chọn
2. Đọc lại bài viết và in chủ đề cùng số từ
3. Cập nhật nội dung bài viết với đoạn văn bản mới
4. Đọc lại và xác nhận số từ đã thay đổi

Đây chính là quy trình mà Content Writer agent sử dụng — lưu, đọc, cập nhật.

In [ ]:
# Bài tập: Điền vào chỗ trống để thao tác với local storage

# 1. Tạo bài viết (điền chủ đề và từ khóa)
result = save_article(___, ___, keywords=___)
my_article = json.loads(result)
my_id = my_article[___]
print(f"\u0110\u00e3 t\u1ea1o b\u00e0i vi\u1ebft: {my_id}")

# 2. \u0110\u1ecdc l\u1ea1i b\u00e0i vi\u1ebft
result = ___(my_id)
content = json.loads(result)
print(f"Ch\u1ee7 \u0111\u1ec1: {content[___]}")
print(f"S\u1ed1 t\u1eeb: {len(content['article_markdown'].split())}")

# 3. C\u1eadp nh\u1eadt n\u1ed9i dung
new_content = content['article_markdown'] + "\n\n## New Section\n\nAdded by the exercise."
result = update_article_content(my_id, ___)
updated = json.loads(result)
print(f"\nSau khi c\u1eadp nh\u1eadt:")
print(f"S\u1ed1 t\u1eeb: {updated[___]}")

<details>
<summary>Bấm để xem đáp án</summary>

```python
# 1. Tạo bài viết
result = save_article("SEO Tips for Small Businesses", "# SEO Tips\n\nHere are some tips for small businesses.", keywords="seo tips, small business")
my_article = json.loads(result)
my_id = my_article["article_id"]
print(f"Đã tạo bài viết: {my_id}")

# 2. Đọc lại bài viết
result = get_article_content(my_id)
content = json.loads(result)
print(f"Chủ đề: {content['topic']}")
print(f"Số từ: {len(content['article_markdown'].split())}")

# 3. Cập nhật nội dung
new_content = content['article_markdown'] + "\n\n## New Section\n\nAdded by the exercise."
result = update_article_content(my_id, new_content)
updated = json.loads(result)
print(f"\nSau khi cập nhật:")
print(f"Số từ: {updated['word_count']}")
```
</details>